# 04 Visualize Results

Visualize Frankfurt airspace complexity outputs from the Gold layer and prepare research-facing figures for the case study.

Primary inputs:

- `adsb_airspace_eddf.gld_airspace.grid_complexity_5m`
- `adsb_airspace_eddf.gld_airspace.complexity_hotspots`
- `adsb_airspace_eddf.gld_airspace.complexity_trend_15m`
- `adsb_airspace_eddf.ref.grid_cells`

## Figure Plan

This notebook generates three core outputs for the Frankfurt MVP:

1. Complexity heatmap across the Frankfurt grid
2. 15-minute complexity trend over time
3. Top hotspot ranking bar chart

It also shows compact table previews to help validate the plotted results.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import yaml

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)

with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)

def parse_int(raw_value: str, *, default: int) -> int:
    text = raw_value.strip()
    return default if not text else int(text)

default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
default_source_run_id = ""
default_top_n = "10"

catalog = default_catalog
source_run_id_raw = default_source_run_id
top_n_raw = default_top_n

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    dbutils.widgets.text("source_run_id", default_source_run_id)
    dbutils.widgets.text("top_n", default_top_n)

    catalog = dbutils.widgets.get("catalog").strip() or default_catalog
    source_run_id_raw = dbutils.widgets.get("source_run_id").strip()
    top_n_raw = dbutils.widgets.get("top_n").strip() or default_top_n

focus_airport = region_config["focus_airport"]
bbox = region_config["regional_bbox"]
top_n = parse_int(top_n_raw, default=10)
if top_n <= 0:
    raise ValueError("top_n must be positive")

grid_complexity_table = f"{catalog}.gld_airspace.grid_complexity_5m"
hotspots_table = f"{catalog}.gld_airspace.complexity_hotspots"
trend_table = f"{catalog}.gld_airspace.complexity_trend_15m"
grid_ref_table = f"{catalog}.ref.grid_cells"
source_run_id_input = source_run_id_raw.strip()

plt.style.use("ggplot")
pd.set_option("display.max_columns", 20)


In [ ]:
if source_run_id_input:
    selected_source_run_id = source_run_id_input
else:
    latest_success_row = spark.sql(
        f"""
        SELECT run_id
        FROM {catalog}.obs.pipeline_run_log
        WHERE pipeline_name = '03_build_complexity_metrics'
          AND status = 'success'
        ORDER BY completed_at DESC
        LIMIT 1
        """
    ).first()
    if latest_success_row is None:
        raise ValueError("No successful Gold run found. Run 03_build_complexity_metrics.ipynb first.")
    selected_source_run_id = latest_success_row["run_id"]

grid_complexity_df = spark.table(grid_complexity_table).where(F.col("run_id") == F.lit(selected_source_run_id))
hotspots_df = spark.table(hotspots_table).where(F.col("run_id") == F.lit(selected_source_run_id))
trend_df = spark.table(trend_table).where(F.col("run_id") == F.lit(selected_source_run_id))
grid_ref_df = spark.table(grid_ref_table).where(F.col("run_id") == F.lit(selected_source_run_id))

grid_rows = grid_complexity_df.count()
hotspot_rows = hotspots_df.count()
trend_rows = trend_df.count()
grid_ref_rows = grid_ref_df.count()

if grid_rows == 0 or hotspot_rows == 0 or trend_rows == 0:
    raise ValueError(f"Gold rows missing for run_id={selected_source_run_id}. Run 03_build_complexity_metrics.ipynb first.")

grid_summary = grid_complexity_df.agg(
    F.countDistinct("grid_id").alias("grid_count"),
    F.countDistinct("window_start").alias("window_count"),
    F.min("window_start").alias("min_window_start"),
    F.max("window_start").alias("max_window_start"),
    F.min("complexity_score").alias("min_complexity_score"),
    F.max("complexity_score").alias("max_complexity_score")
).first()

run_summary = {
    "catalog": catalog,
    "focus_airport": focus_airport,
    "source_run_id": selected_source_run_id,
    "top_n": top_n,
    "grid_rows": grid_rows,
    "hotspot_rows": hotspot_rows,
    "trend_rows": trend_rows,
    "grid_ref_rows": grid_ref_rows,
    "grid_count": int(grid_summary["grid_count"]),
    "window_count": int(grid_summary["window_count"]),
    "min_window_start": str(grid_summary["min_window_start"]),
    "max_window_start": str(grid_summary["max_window_start"]),
    "min_complexity_score": float(grid_summary["min_complexity_score"]),
    "max_complexity_score": float(grid_summary["max_complexity_score"]),
    "bbox": bbox,
}

run_summary


In [ ]:
heatmap_df = (
    grid_complexity_df
    .groupBy("grid_id")
    .agg(
        F.avg("complexity_score").alias("avg_complexity_score"),
        F.max("complexity_score").alias("peak_complexity_score"),
        F.avg("aircraft_count").alias("avg_aircraft_count")
    )
    .join(grid_ref_df.select("grid_id", "center_latitude", "center_longitude", "min_latitude", "max_latitude", "min_longitude", "max_longitude"), on="grid_id", how="inner")
    .orderBy(F.col("avg_complexity_score").desc())
).toPandas()

trend_plot_df = trend_df.orderBy("window_start").toPandas()
hotspots_plot_df = hotspots_df.orderBy("ranking").limit(top_n).toPandas()
grid_preview_df = grid_complexity_df.orderBy(F.col("complexity_score").desc()).limit(20).toPandas()

if heatmap_df.empty or trend_plot_df.empty or hotspots_plot_df.empty:
    raise ValueError("Visualization inputs are empty after converting to pandas.")

trend_plot_df["window_start"] = pd.to_datetime(trend_plot_df["window_start"])
hotspots_plot_df = hotspots_plot_df.sort_values("ranking", ascending=False)

preview_payload = {
    "heatmap_rows": int(len(heatmap_df)),
    "trend_rows": int(len(trend_plot_df)),
    "hotspot_rows": int(len(hotspots_plot_df)),
}

preview_payload


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    heatmap_df["center_longitude"],
    heatmap_df["center_latitude"],
    c=heatmap_df["avg_complexity_score"],
    s=170,
    marker="s",
    cmap="YlOrRd",
    alpha=0.85,
    edgecolors="none",
)
ax.set_title(f"Frankfurt Complexity Heatmap\nrun_id={selected_source_run_id}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(float(bbox["min_longitude"]), float(bbox["max_longitude"]))
ax.set_ylim(float(bbox["min_latitude"]), float(bbox["max_latitude"]))
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Average Complexity Score")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trend_plot_df["window_start"], trend_plot_df["avg_complexity_score"], marker="o", linewidth=2, label="Average")
ax.plot(trend_plot_df["window_start"], trend_plot_df["peak_complexity_score"], marker="s", linewidth=2, label="Peak")
ax.set_title(f"Frankfurt 15-minute Complexity Trend\nrun_id={selected_source_run_id}")
ax.set_xlabel("Window Start (UTC)")
ax.set_ylabel("Complexity Score")
ax.legend()
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(hotspots_plot_df["grid_id"], hotspots_plot_df["avg_complexity_score"], color="#d95f02")
ax.set_title(f"Top {top_n} Frankfurt Complexity Hotspots\nrun_id={selected_source_run_id}")
ax.set_xlabel("Average Complexity Score")
ax.set_ylabel("Grid ID")
plt.tight_layout()
plt.show()


In [ ]:
display(spark.createDataFrame(heatmap_df.sort_values("avg_complexity_score", ascending=False).head(20)))
display(spark.createDataFrame(trend_plot_df))
display(spark.createDataFrame(hotspots_plot_df.sort_values("ranking").head(top_n)))
display(spark.createDataFrame(grid_preview_df))


In [ ]:
final_summary = {
    "status": "success",
    "source_run_id": selected_source_run_id,
    "grid_rows": grid_rows,
    "hotspot_rows": hotspot_rows,
    "trend_rows": trend_rows,
    "grid_count": run_summary["grid_count"],
    "window_count": run_summary["window_count"],
    "top_n": top_n,
    "heatmap_points": preview_payload["heatmap_rows"],
}

final_summary
